# Simple CUDA Demo

This notebook demonstrates a basic CUDA-accelerated operation using PyTorch on a synthetic ECG-like signal.
It is designed to run quickly in environments like Kaggle or Google Colab without requiring large datasets.

**Goal**: Apply a simple low-pass filter (moving average) using GPU acceleration and compare performance to CPU.


In [ ]:
# Check if CUDA is available
import torch
import numpy as np
import time

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'CUDA is available. Using device: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('CUDA is not available. Using CPU.')


In [ ]:
# Create a synthetic ECG-like signal (larger size to see GPU benefit)
fs = 500  # Hz
duration = 30  # seconds
n_samples = int(fs * duration)
n_leads = 12  # standard ECG leads
t = np.linspace(0, duration, n_samples)
# Simulate multi-lead ECG: each lead is a combination of sine waves with different phases and noise
ecg = np.zeros((n_samples, n_leads), dtype=np.float32)
for lead in range(n_leads):
    # Different frequency components for each lead
    ecg[:, lead] = (np.sin(2*np.pi*1.0*t) + 
                    0.5*np.sin(2*np.pi*20.0*t + lead*0.5) + 
                    0.2*np.sin(2*np.pi*60.0*t + lead*0.2) + 
                    0.15*np.random.randn(n_samples))

print(f'Signal shape: {ecg.shape} (samples, leads)')
print(f'First 5 samples of first lead: {ecg[:5, 0]}')


In [ ]:
# Move signal to device
ecg_tensor = torch.from_numpy(ecg).to(device)

# Define a simple low-pass filter (Moving average) using convolution on GPU
def moving_average_filter(signal, kernel_size=5):
    # Create kernel
    kernel = torch.ones(kernel_size, device=device) / kernel_size
    # Pad signal to maintain size
    padding = kernel_size // 2
    padded = torch.nn.functional.pad(signal.unsqueeze(0).unsqueeze(0), (padding, padding), mode='reflect')
    # Convolve
    filtered = torch.nn.functional.conv1d(padded, kernel.unsqueeze(0).unsqueeze(0))
    return filtered.squeeze()


In [ ]:
# Define a simple low-pass filter (Moving average) using convolution on GPU
def moving_average_filter(signal, kernel_size=5):
    # signal shape: (samples, leads) -> we treat each lead as a separate channel
    # We'll loop over leads and apply 1D convolution to each lead's signal.
    kernel = torch.ones(kernel_size, device=device) / kernel_size
    padding = kernel_size // 2
    filtered = torch.zeros_like(signal)
    for lead in range(signal.shape[1]):
        lead_signal = signal[:, lead].unsqueeze(0).unsqueeze(0)  # (1,1,samples)
        padded = torch.nn.functional.pad(lead_signal, (padding, padding), mode='reflect')
        lead_filtered = torch.nn.functional.conv1d(padded, kernel.unsqueeze(0).unsqueeze(0))
        filtered[:, lead] = lead_filtered.squeeze()
    return filtered


In [ ]:
# Apply filter on GPU and time it
start = time.time()
ecg_filtered_gpu = moving_average_filter(ecg_tensor, kernel_size=11)
if device.type == 'cuda':
    torch.cuda.synchronize()  # Wait for GPU to finish
gpu_time = time.time() - start

print(f'GPU Filtering took {gpu_time:.4f} seconds on {device}.')

# Apply filter on CPU for comparison
ecg_tensor_cpu = ecg_tensor.cpu()
start = time.time()
ecg_filtered_cpu = moving_average_filter(ecg_tensor_cpu, kernel_size=11)
cpu_time = time.time() - start
print(f'CPU Filtering took {cpu_time:.4f} seconds on cpu.')

if gpu_time > 0:
    speedup = cpu_time / gpu_time if gpu_time > 0 else float('inf')
    print(f'Speedup (CPU/GPU): {speedup:.2f}x')
else:
    print('GPU time zero, cannot compute speedup.')


---

**Notes:**
- This is a deliberately simple demonstration of CUDA usage; real ECG preprocessing would involve more complex filtering.
- The moving average filter is implemented via 1D convolution, which is efficiently parallelized on GPU.
- We increased the signal size (30 seconds at 500 Hz, 12 leads) to make GPU acceleration benefits visible.
- In environments like Kaggle or Google Colab, ensure GPU accelerator is enabled (Runtime → Change runtime type → GPU).
- For very small signals, CPU may be faster due to GPU launch overhead; with larger data, GPU wins.
- The speedup depends on the hardware and signal size.
